In [1]:
# %load_ext autoreload
# %autoreload 2


import random

### My libraries (not pip)

try:
    import mathlib as Mathlib   #< c++ version (much faster training). Must be compiled first
    print("Using C++ compiled mathlib!")
except ImportError:
    import Mathlib
    print("Using python mathlib as the C++ compiled library is not avaialble!")
import Activation
import Loss
import Optimizer
import Accuracy
import Neuron
import Layer
import Batch

# import importlib
# importlib.reload(Mathlib)    #< cpp library
# importlib.reload(Activation)
# importlib.reload(Loss)
# importlib.reload(Optimizer)
# importlib.reload(Accuracy)
# importlib.reload(Layer)
# importlib.reload(Batch)

from Layer import Layer
from Batch import Batch


Using python mathlib as the C++ compiled library is not avaialble!


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



## Functions to run a forward and backward pass on a super simple network with two layer totaling 9 neurons

In [2]:
random.seed(1)

def createInputs(inputCount=10):
    return([Mathlib.randomNumber(-10, 10, 2) for _ in range(inputCount)])

inputsCount = 5
target = 2
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
layer2NeuronCount = 5

layer1 = Layer(inputsCount, layer1NeuronCount, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
layer2 = Layer(layer1NeuronCount, layer2NeuronCount, activationFunc=Activation.Pass)   #< output layer returns raw logits for softmax

accuracyCalculator = Accuracy.Accuracy_hard()

def stepForward():
    layer1Output = layer1.forward(inputs)
    layer2Output = layer2.forward(layer1Output)
    softmaxOutput = Activation.ProtectedSoftmax.forward(layer2Output)
    error = Loss.Entropy.forwardSparse(softmaxOutput, target)
    accuracy = accuracyCalculator.epoch(softmaxOutput, target)    #< we want to maximize the 2nd output
    return(layer1Output, layer2Output, softmaxOutput, error, accuracy)

def stepBackward(softmaxOutput):
    d_error  = Loss.Entropy.backwardSparse(softmaxOutput, target)
    d_softmax = Activation.Softmax.backward(softmaxOutput)
    d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax) #< not that we can calculate the cross entropy function much simpler using the shortcut backwardCrossEntropy, but I choose to take the long way in this example because its easier to understand
    d_layer2 = layer2.backward(d_crossEntropy)
    d_layer1 = layer1.backward(d_layer2)
    return(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

def printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy):
    print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer1NeuronCount} biases")
    print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
    print()
    print(f"WEIGHTS: {layer2.getWeights()}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
    print(f"BIASES: {layer2.getBiases()}\n    this layer should have {layer2NeuronCount} biases")
    print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
    print("SOFTMAX: ", softmaxOutput)
    print("TARGET: ", target)
    print(f"ERROR: {error} should be between 0 and 16.12")
    print("ACCURACY: ", accuracy)

def printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1):
    print("d_ERROR: ", d_error)
    print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(d_softmax)}")
    print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
    print("d_LAYER 2: ", d_layer2)
    print("d_LAYER 1: ", d_layer1)

layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

INPUTS: [-7.31, 6.95, 5.28, -4.9, -0.09]
    This layer should have 5 inputs
WEIGHTS: [[-0.09, 0.27, 0.52, -0.73, -0.84], [0.6, -0.12, 0.47, -0.89, -0.1], [0.4, -0.49, 0.8, 0.72, -0.84], [-0.85, 0.07, 0.79, -0.21, -0.51]]
    this layer should have 4 weights with 5 values in them
BIASES: [0.0, 0.0, 0.0, 0.0]
    this layer should have 4 biases
RESULT: [8.932599999999999, 1.6316000000000015, 0.0, 11.946100000000001]
     this layer should have 4 outputs

WEIGHTS: [[-0.08, -0.47, -0.28, -0.06], [-0.0, -0.27, -0.27, -0.28], [-0.04, -0.21, -0.48, 0.34], [0.06, 0.14, -0.31, 0.49], [0.36, -0.38, -0.17, 0.22]]
    this layer should have 5 weights with 4 values in them
BIASES: [0.0, 0.0, 0.0, 0.0, 0.0]
    this layer should have 5 biases
RESULT: [-2.198226000000001, -3.7854400000000012, 3.3617340000000007, 6.617969, 5.22387]
     this layer should have 5 outputs
SOFTMAX:  [0.00011525899123100589, 2.356983446623867e-05, 0.029945720160598762, 0.7771406998567363, 0.19277475115696757]
TARGET:  2
E

### Optimizing the neural network

In [3]:
optimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
for i in range(5000):
    layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
    d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
    #printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
    #printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)
    optimizer.update(layer2)
    optimizer.update(layer1)
    print(f"Epoch {i+1}: Error: {error}, accuracy: {accuracy}")

Epoch 1: Error: 3.508368864110955, accuracy: 0.0
Epoch 2: Error: 0.6599484227831312, accuracy: 0.3333333333333333
Epoch 3: Error: 0.17084090965057475, accuracy: 0.5
Epoch 4: Error: 0.10385946822308692, accuracy: 0.6
Epoch 5: Error: 0.07519647924273754, accuracy: 0.6666666666666666
Epoch 6: Error: 0.059061829980337543, accuracy: 0.7142857142857143
Epoch 7: Error: 0.04866439943225191, accuracy: 0.75
Epoch 8: Error: 0.04139008927776971, accuracy: 0.7777777777777778
Epoch 9: Error: 0.036009386066085655, accuracy: 0.8
Epoch 10: Error: 0.031865419856969376, accuracy: 0.8181818181818182
Epoch 11: Error: 0.02857453147142794, accuracy: 0.8333333333333334
Epoch 12: Error: 0.025897287532926052, accuracy: 0.8461538461538461
Epoch 13: Error: 0.023676392125197242, accuracy: 0.8571428571428571
Epoch 14: Error: 0.021804182117171736, accuracy: 0.8666666666666667
Epoch 15: Error: 0.020204449416890424, accuracy: 0.875
Epoch 16: Error: 0.018821718053138613, accuracy: 0.8823529411764706
Epoch 17: Error: 0.

In [ ]:
import json
import time
start = time.time()

with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")


try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    spiralLayer1 = Batch(inputCount=32*32, neuronCount=32, activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(inputCount=32, neuronCount=2, activationFunc=Activation.Pass)

    spiralLayer1.weights = savedModel["layer1"]["weights"]
    spiralLayer1.biases = savedModel["layer1"]["biases"]
    spiralLayer2.weights = savedModel["layer2"]["weights"]
    spiralLayer2.biases = savedModel["layer2"]["biases"]
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")
    spiralLayer1 = Batch(
    # spiralLayer1 = Layer(
                        inputCount=32*32,
                        neuronCount=32,   #< output features
                        activationFunc=Activation.LeakyReLU)
    spiralLayer2 = Batch(
    # spiralLayer2 = Layer(
                        inputCount=32,
                        neuronCount=2,
                        activationFunc=Activation.Pass)
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    BATCH_SIZE = 32
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = spiralLayer1.forward(X_batch)                              #< returns a list of lists of outputs
        layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
        # print(layer2Output)
        # error = softmaxCrossEntropy.forward(layer2Output, y_batch)              #< returns mean error (layer mode)
        error = softmaxCrossEntropy.forward_batch(layer2Output, y_batch)          #< returns mean error (batch mode)
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward_batch()
        # d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = spiralLayer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = spiralLayer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(spiralLayer2)
        spiralOptimizer.update(spiralLayer1)

    print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")  #< Normal to bounce around at the start
    
    ## Save model
    modelState = {
        "layer1": {
            "weights": spiralLayer1.getWeights(),
            "biases": spiralLayer1.getBiases()
        },
        "layer2": {
            "weights": spiralLayer2.getWeights(),
            "biases": spiralLayer2.getBiases()
        }
    }

    if epoch%10 == 0:
        with open("trainedModel.json", "w") as f:
            json.dump(modelState, f, indent=4)

end = time.time()
print(f"Training took: {end-start} seconds")


Converting data to hilbert space
Done/1000
Successfully loaded 'trainedModel.json'!
Epoch 0: Error: 0.7417815805661028, accuracy: 0.525
Epoch 1: Error: 0.7394137641292753, accuracy: 0.5285
Epoch 2: Error: 0.7381705198154818, accuracy: 0.5353333333333333
Epoch 3: Error: 0.7372493005187772, accuracy: 0.54025
Epoch 4: Error: 0.7342790450823257, accuracy: 0.544
Epoch 5: Error: 0.73482733707569, accuracy: 0.5486666666666666
Epoch 6: Error: 0.7328954886246805, accuracy: 0.5527142857142857
Epoch 7: Error: 0.730859447004893, accuracy: 0.556375
Epoch 8: Error: 0.7294022750325303, accuracy: 0.5598888888888889
Epoch 9: Error: 0.7278067765943483, accuracy: 0.5627
Epoch 10: Error: 0.7243873854656296, accuracy: 0.5651818181818182
Epoch 11: Error: 0.7223207942228907, accuracy: 0.5675
Epoch 12: Error: 0.7203140343148058, accuracy: 0.5698461538461539
Epoch 13: Error: 0.7181843597268655, accuracy: 0.5725
Epoch 14: Error: 0.7149593409243896, accuracy: 0.575
Epoch 15: Error: 0.7126244464321947, accuracy: 

The raw Python model takes slightly less than 90 minutes to fully train.
Python and C++ model takes about 30 mins to train to 90% accuracy and 64 minutes to fully train(95.74% accuracy) Thats 1.4x the speed!

In [ ]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()